# 3D tif画像から基本的な空隙率を計算するNotebook

このNotebookでは、3D SEM / FIB-SEMで得られた3D tif画像から、最も基本的な構造指標である空隙率を計算します。
まず画像を二値化し、固体相と孔相を分離します。
その後、全体空隙率と、X/Y/Z方向のslice-by-slice空隙率を計算します。

ここでは画像配列のshapeを **(Z, Y, X)**、すなわち「Z方向のスライス枚数、高さ、幅」と仮定します。

## 注意：二値化条件の目視確認が重要です

二値化のしきい値と、孔が暗いか明るいかの設定は、空隙率に大きく影響します。
そのため、計算値を見る前に、必ず中央スライスと複数スライスのpore maskを目視確認してください。

特にFIB-SEM画像では、カーテニング、チャージアップ、輝度ドリフト、再付着などにより、単純なOtsu二値化が不適切な場合があります。
その場合は、`PORE_IS_DARK`、`USE_MANUAL_THRESHOLD`、`MANUAL_THRESHOLD`、フィルタ条件、または前処理を見直してください。

## 1. パラメータ設定

まず、ユーザーが変更する可能性の高いパラメータをまとめます。
`PORE_IS_DARK` は特に重要です。SEM画像では孔が黒く、固体が明るいことが多いため、デフォルトは `True` にしています。

In [ ]:
# 必要に応じて実行
# %pip install numpy matplotlib tifffile scipy scikit-image pandas

from pathlib import Path

# 入力3D tifファイル
INPUT_TIF = "your_3d_sem.tif"

# 結果の出力フォルダ
OUTPUT_DIR = "porosity_output"

# 孔が暗い画像なら True、孔が明るい画像なら False
PORE_IS_DARK = True

# フィルタ設定
USE_MEDIAN_FILTER = True
MEDIAN_SIZE = 2

# 必要なら手動しきい値を使えるようにする
USE_MANUAL_THRESHOLD = False
MANUAL_THRESHOLD = 128

input_path = Path(INPUT_TIF)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

## 2. tif読み込み

`tifffile.imread()` で3D tifスタック画像を読み込みます。
このNotebookでは、読み込んだ画像のshapeが **(Z, Y, X)** である前提で処理します。
読み込み後に、shape、dtype、輝度範囲、平均輝度を確認します。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile
from scipy.ndimage import median_filter
from skimage.filters import threshold_otsu
import pandas as pd

if not input_path.exists():
    raise FileNotFoundError(
        f"入力ファイルが見つかりません: {input_path}\n"
        "INPUT_TIF を実際の3D .tif / .tiff ファイルのパスに変更してください。"
    )

img = tifffile.imread(input_path)

if img.ndim != 3:
    raise ValueError(
        f"3D画像を想定していますが、読み込んだ配列は {img.ndim} 次元です。shape={img.shape}"
    )

z_size, y_size, x_size = img.shape

print("=== Image information ===")
print(f"shape (Z, Y, X): {img.shape}")
print(f"dtype: {img.dtype}")
print(f"min: {img.min()}")
print(f"max: {img.max()}")
print(f"mean: {img.mean():.4f}")

## 3. 中央スライス表示

Z方向の中央スライスを表示し、同時に輝度ヒストグラムを確認します。
ヒストグラムは、しきい値処理が妥当かどうかを判断するための基本情報になります。

In [ ]:
center_z = z_size // 2
center_slice = img[center_z]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(center_slice, cmap="gray")
axes[0].set_title(f"Original center slice (Z={center_z})")
axes[0].axis("off")

axes[1].hist(img.ravel(), bins=256, color="gray")
axes[1].set_title("Intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Voxel count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. ノイズ除去

必要に応じて、`scipy.ndimage.median_filter` によるメディアンフィルタをかけます。
メディアンフィルタは孤立ノイズを減らすのに有効ですが、過度に強くすると微細な孔構造を消す可能性があります。
`USE_MEDIAN_FILTER = False` の場合は、元画像をそのまま使います。

In [ ]:
if USE_MEDIAN_FILTER:
    img_filt = median_filter(img, size=MEDIAN_SIZE)
    print(f"Median filter applied: size={MEDIAN_SIZE}")
else:
    img_filt = img
    print("Median filter was skipped. Original image is used for binarization.")

## 5. Otsu二値化

`skimage.filters.threshold_otsu` を使って、Otsu法でしきい値を決定します。
`USE_MANUAL_THRESHOLD = True` の場合は、`MANUAL_THRESHOLD` を使います。

孔が暗い画像では `img_filt < threshold` を孔相、孔が明るい画像では `img_filt > threshold` を孔相とします。
固体相は `solid = ~pore` として作成します。

In [ ]:
if USE_MANUAL_THRESHOLD:
    threshold = MANUAL_THRESHOLD
    print(f"Manual threshold is used: {threshold}")
else:
    threshold = threshold_otsu(img_filt)
    print(f"Otsu threshold: {threshold}")

if PORE_IS_DARK:
    pore = img_filt < threshold
else:
    pore = img_filt > threshold

solid = ~pore

print(f"PORE_IS_DARK: {PORE_IS_DARK}")
print(f"pore dtype: {pore.dtype}, solid dtype: {solid.dtype}")

## 6. 二値化結果の確認

中央スライスについて、元画像、フィルタ後画像、pore mask、solid maskを並べて表示します。
pore maskでは孔相が白く見えるように表示します。

さらに、Z方向に等間隔で5枚程度のスライスを表示し、二値化が破綻していないか確認します。
ここで見た目が不自然な場合は、空隙率の数値を解釈する前にパラメータを調整してください。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img[center_z], cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(img_filt[center_z], cmap="gray")
axes[1].set_title("Filtered")
axes[1].axis("off")

axes[2].imshow(pore[center_z], cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Pore mask")
axes[2].axis("off")

axes[3].imshow(solid[center_z], cmap="gray", vmin=0, vmax=1)
axes[3].set_title("Solid mask")
axes[3].axis("off")

plt.tight_layout()
plt.show()

# 等間隔で最大5枚のZスライスを確認します
n_preview = min(5, z_size)
preview_indices = np.linspace(0, z_size - 1, n_preview, dtype=int)

fig, axes = plt.subplots(2, n_preview, figsize=(3 * n_preview, 6))
if n_preview == 1:
    axes = np.array(axes).reshape(2, 1)

for col, z in enumerate(preview_indices):
    axes[0, col].imshow(img[z], cmap="gray")
    axes[0, col].set_title(f"Original Z={z}")
    axes[0, col].axis("off")

    axes[1, col].imshow(pore[z], cmap="gray", vmin=0, vmax=1)
    axes[1, col].set_title(f"Pore mask Z={z}")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

## 7. 全体空隙率の計算

3D視野全体の空隙率と固体率を計算します。
`pore` は孔相のvoxelが `True` であるbool配列なので、`pore.mean()` が孔相voxelの割合、すなわち空隙率になります。

In [ ]:
porosity = pore.mean()
solid_fraction = solid.mean()

print(f"Porosity: {porosity:.4f}")
print(f"Porosity [%]: {porosity * 100:.2f}")
print(f"Solid fraction [%]: {solid_fraction * 100:.2f}")

## 8. Z方向のslice-by-slice空隙率

各Zスライスごとに、Y-X平面内の空隙率を計算します。
厚み方向の不均一性、加工影響、局所的な詰まりなどを確認するために使います。

In [ ]:
porosity_z = pore.mean(axis=(1, 2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(z_size), porosity_z, marker="o", linewidth=1)
ax.axhline(porosity, color="red", linestyle="--", label=f"Mean = {porosity:.4f}")
ax.set_xlabel("Z slice")
ax.set_ylabel("Porosity")
ax.set_title("Z-direction slice-by-slice porosity")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 9. X/Y方向のslice-by-slice空隙率

X方向、Y方向についても、各断面ごとの空隙率プロファイルを計算します。
横方向の視野内ムラや、画像端の影響を確認するために使います。

In [ ]:
porosity_y = pore.mean(axis=(0, 2))
porosity_x = pore.mean(axis=(0, 1))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(x_size), porosity_x, label="X profile", linewidth=1)
ax.plot(np.arange(y_size), porosity_y, label="Y profile", linewidth=1)
ax.axhline(porosity, color="red", linestyle="--", label=f"Mean = {porosity:.4f}")
ax.set_xlabel("Slice index")
ax.set_ylabel("Porosity")
ax.set_title("X/Y-direction slice-by-slice porosity")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 10. 結果保存

中央スライス画像、pore mask、Z方向プロファイル、X/Y方向プロファイルをPNGで保存します。
また、全体空隙率のsummaryと、各方向の空隙率プロファイルをCSVで保存します。

In [ ]:
# --- 画像保存 ---
plt.figure(figsize=(6, 6))
plt.imshow(img[center_z], cmap="gray")
plt.title(f"Original center slice (Z={center_z})")
plt.axis("off")
plt.tight_layout()
plt.savefig(output_dir / "center_slice_original.png", dpi=200, bbox_inches="tight")
plt.close()

plt.figure(figsize=(6, 6))
plt.imshow(pore[center_z], cmap="gray", vmin=0, vmax=1)
plt.title(f"Pore mask center slice (Z={center_z})")
plt.axis("off")
plt.tight_layout()
plt.savefig(output_dir / "center_slice_pore_mask.png", dpi=200, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(z_size), porosity_z, marker="o", linewidth=1)
ax.axhline(porosity, color="red", linestyle="--", label=f"Mean = {porosity:.4f}")
ax.set_xlabel("Z slice")
ax.set_ylabel("Porosity")
ax.set_title("Z-direction slice-by-slice porosity")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / "porosity_z_profile.png", dpi=200, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(x_size), porosity_x, label="X profile", linewidth=1)
ax.plot(np.arange(y_size), porosity_y, label="Y profile", linewidth=1)
ax.axhline(porosity, color="red", linestyle="--", label=f"Mean = {porosity:.4f}")
ax.set_xlabel("Slice index")
ax.set_ylabel("Porosity")
ax.set_title("X/Y-direction slice-by-slice porosity")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(output_dir / "porosity_xy_profile.png", dpi=200, bbox_inches="tight")
plt.close()

# --- CSV保存 ---
summary = pd.DataFrame([
    {
        "input_file": str(input_path),
        "shape_z": z_size,
        "shape_y": y_size,
        "shape_x": x_size,
        "dtype": str(img.dtype),
        "threshold": threshold,
        "pore_is_dark": PORE_IS_DARK,
        "porosity": porosity,
        "porosity_percent": porosity * 100,
        "solid_fraction": solid_fraction,
        "solid_fraction_percent": solid_fraction * 100,
    }
])
summary.to_csv(output_dir / "porosity_summary.csv", index=False)

pd.DataFrame({"z_slice": np.arange(z_size), "porosity": porosity_z}).to_csv(
    output_dir / "porosity_z_profile.csv", index=False
)
pd.DataFrame({"x_slice": np.arange(x_size), "porosity": porosity_x}).to_csv(
    output_dir / "porosity_x_profile.csv", index=False
)
pd.DataFrame({"y_slice": np.arange(y_size), "porosity": porosity_y}).to_csv(
    output_dir / "porosity_y_profile.csv", index=False
)

print(f"Results were saved to: {output_dir.resolve()}")
print("Saved files:")
for path in sorted(output_dir.iterdir()):
    print(f"- {path.name}")

## 11. 解釈

最後に、計算された平均空隙率と、方向別プロファイルを見るときの基本的な注意を表示します。

In [ ]:
print("=== Interpretation ===")
print(f"この3D視野の平均空隙率は {porosity*100:.2f} % です。")
print("Z方向プロファイルに大きな変動がある場合、厚み方向の層差、塗工ムラ、FIB加工影響、または局所的な詰まりが疑われます。")
print("この空隙率は観察視野内の値であり、材料全体の代表値とみなすには複数視野またはRVE確認が必要です。")